In [8]:
from setup import *
from datetime import datetime as dt

In [2]:
entities = pd.read_csv("./data/metadata/all_entities.csv")

In [9]:
datestring = dt.now().strftime("%Y-%m-%d")

# Trefferliste abrufen

Mit diesem Notebook rufen wir für eine Liste an Entities die Treffer für eine Liste an Suchbegriffen ab und strukturieren diese.

## Search Terms definieren

Über die Keyword-Suche können wir alle Treffer zu bestimmten Suchbegriffen finden. Für die Erstellung der Suchbegriff-Liste bietet es sich an, zunächst über das Dashboard Inhalte zu explorieren, und/oder Clara zu fragen.



In [6]:
entity_ids = entities["id"].to_list()
search_terms_wp = ["Wärmeplanung", "Wärmeplan", "Fernwärme", "Fernwärmenetz", "Wärmenetz", "Wärmeversorgung", "Wärmeversorgungskonzept", "Wärmeversorgungskonzepte", "Wärmeplanungsgesetz"]
search_terms_autofrei = ["autofrei", "autofreie Stadt", "autofreies Stadtzentrum", "autofreie Innenstadt", "autofreie Innenstadtbereiche", "autofreie Innenstadtbereiche", "autofreie Innenstadtbereiche", "autofreie Innenstadtbereiche"]
search_terms_autoarm = ["autoarm", "autoarme Stadt", "autoarmes Stadtzentrum", "autoarme Innenstadt", "autoarme Innenstadtbereiche", "autoarme Innenstadtbereiche", "autoarme Innenstadtbereiche", "autoarme Innenstadtbereiche"]

search_terms_hitzeschutz = [ "Hitzeschutz", "Hitzeaktion", "Kühlraum", "Kühlräume", "Hitzevorsorge", "Hitzewarn" ]
thema = "hitzeschutz"
search_terms = search_terms_hitzeschutz
search_string = " OR ".join(search_terms)

In [4]:
# Alle Items sammeln
all_items = []

# Paginierungsparameter
limit = 499
request_count = 0

print(f"\n🔍 Durchsuche {len(entity_ids)} Entity IDs...")

offset = 0
total_items = None

print("Using shared setup defaults for API auth and base URL")

while True:
    response = poliscope_request(
        "GET",
        "/search/scan",
        params={
            "q": search_string,
            "limit": limit,
            "offset": offset,
            "entityIds": ["03*"]  # Ganz Niedersachsen
        },
        timeout=10.0,
    )

    request_count += 1

    data = response.json()
    items = data.get("data", [])

    meta = data.get("meta", {})
    pagination = meta.get("pagination", {})
    total_items = pagination.get("total", 0)

    if not items:
        print("  → Keine weiteren Treffer.")
        break

    all_items.extend(items)
    offset += len(items)
    print(f"  → Downloaded {offset} / {total_items} items (Request #{request_count})")

    if offset >= total_items or len(items) < limit:
        break

print(f"\n✓ FERTIG! Insgesamt {len(all_items)} items abgerufen.")


🔍 Durchsuche 16079 Entity IDs...
Using shared setup defaults for API auth and base URL
  → Downloaded 499 / 3382 items (Request #1)
  → Downloaded 998 / 3382 items (Request #2)
  → Downloaded 1497 / 3382 items (Request #3)
  → Downloaded 1996 / 3382 items (Request #4)
  → Downloaded 2495 / 3382 items (Request #5)
  → Downloaded 2994 / 3382 items (Request #6)
  → Downloaded 3382 / 3382 items (Request #7)

✓ FERTIG! Insgesamt 3382 items abgerufen.


In [5]:
all_items_df = pd.DataFrame(all_items)
all_items_df


,id,chunkType,groupKey,groupType,agendaItemId,documentId,proposalId,entityId,entityLevel,date
0,document:3c42ab60-6d55-45b9-96cb-deae734b32f4:214,document,proposal:215029b7-c60c-42d3-b994-ca31ab91bbad,proposal,349f050c-5c5e-47a3-80d7-9db60b05cbc5,3c42ab60-6d55-45b9-96cb-deae734b32f4,215029b7-c60c-42d3-b994-ca31ab91bbad,03402,40,2024-12-05T18:00:00
1,document:9a64071d-b6cc-4eed-890f-bc293819584c:298,document,meeting:380ad660-0ec1-438d-8829-1da8466ead70,meeting,f77a23e2-a30f-4b75-971d-ecc6b134b424,9a64071d-b6cc-4eed-890f-bc293819584c,NaN,03405,40,2025-06-04T14:30:00
2,document:6ea16c02-decc-4577-ba1b-c197bd908791:6,document,proposal:11c9a57d-547c-44d8-85ec-616c1c2c3c5c,proposal,01b5e03b-5423-4682-9104-5f870af4ec4b,6ea16c02-decc-4577-ba1b-c197bd908791,11c9a57d-547c-44d8-85ec-616c1c2c3c5c,032410009,50,2025-11-11T18:00:00
3,document:91c91acf-1bc5-49dc-9de8-0c4f66939256:32,document,proposal:dba43975-2322-4e98-929c-5290ed1df04d,proposal,a80e9d0e-dea6-4263-9131-1f1808d33514,91c91acf-1bc5-49dc-9de8-0c4f66939256,dba43975-2322-4e98-929c-5290ed1df04d,034540032,50,2024-10-24T16:01:00
4,document:f1f81c5f-1edb-466b-a794-d625ef45d6a6:186,document,meeting:e1f59738-26fc-4c2a-8de5-c0d3d276d789,meeting,9524fcb6-fc24-47d6-b8b9-42cf07ab1c36,f1f81c5f-1edb-466b-a794-d625ef45d6a6,NaN,032520003,50,2025-05-13T18:00:00
...,...,...,...,...,...,...,...,...,...,...
3377,document:95e06571-8bc9-41b3-8823-3e85495190ff:104,document,meeting:242df0f8-cf84-4574-baca-269b25ab7ea0,meeting,cf4c6722-038b-4810-891e-50600fdcced0,95e06571-8bc9-41b3-8823-3e85495190ff,NaN,033610005005,60,2025-07-14T19:00:00
3378,document:b029d88a-c45f-4228-9f26-fe104d463ca8:4,document,meeting:bb734d49-312e-4324-b2da-8b250002acc6,meeting,757b3dba-24e0-45c5-8bec-760a5b7d1471,b029d88a-c45f-4228-9f26-fe104d463ca8,NaN,03403,40,2024-05-22T17:01:00
3379,document:0070496c-fc82-45f8-913e-5c0e859ba841:2,document,meeting:c4fcd465-3184-4ee6-8723-678c34cfdf85,meeting,de6e3a35-7cbb-4e17-8bd0-0c1e75e0416c,0070496c-fc82-45f8-913e-5c0e859ba841,NaN,03401,40,2025-09-30T17:02:00
3380,document:5eed6aa7-448f-492c-bcc9-070c920a1d71:252,document,meeting:a4ce9ec0-cce7-4bf7-a1df-9247f82da403,meeting,e72f5666-32a2-43ff-ae85-e2209f3db86d,5eed6aa7-448f-492c-bcc9-070c920a1d71,NaN,03103,40,2025-12-10T16:00:00


In [10]:
all_items_df.to_csv(f"./data/raw/{datestring}_{thema}_items.csv", index=False)

highlights: Sind die Character Positions in dem text string

agendaItemID ist die Verknüpfung zu einer Sitzung

vmtl ist proposalID leer, wenns keins gibt